In [1]:
import bs4
import lxml
import pandas as pd
import urllib
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
import urllib.parse

import re
from bs4 import SoupStrainer

from urllib import request
from io import StringIO
import time
from matplotlib import pyplot as plt

import pickle as pckl



# Scraping sujets articles
#### Parcourir les archives du journal FAZ


In [ ]:
only_filter_count = SoupStrainer(class_="filter-count")
def get_faz_mentions(subject, start_year=1993,end_year = 2024,sleep=0):
    subject = subject.replace(" ","+")
    mentions = []
    years = []
    for year in range(start_year,end_year+1):
        url = f"https://fazarchiv.faz.net/faz-portal/faz-archiv?q={subject}&source=&max=10&sort=&offset=0&&_ts={int(time.time())}&DT_from=01.01.{year}&DT_to=01.01.{year+1}&timeFilterType=0#hitlist"
        table_request_text = request.urlopen(url).read()
        site = bs4.BeautifulSoup(table_request_text, "html.parser",parse_only=only_filter_count)
        site = str(site)
        site = site.replace('.','')
        filter_results = re.findall(pattern=r'(?<=\()\d+(?=\))',string=str(site))
        if len(filter_results) > 0:
            number_of_mentions = max([int(x) for x in filter_results])
        else:
            number_of_mentions = 0
        mentions.append(number_of_mentions)
        years.append(year)
        time.sleep(sleep)
    return pd.DataFrame({'Year':years, 'Mentions': mentions}).set_index('Year')

### Parcourir LeMonde

In [4]:
def get_lemonde_mentions(subject, start_year=2010,end_year = 2024,sleep=0):
    subject = subject.replace(" ","+")
    only_page_count_le_monde = SoupStrainer(class_="river__pagination river__pagination--page-search ")
    only_article_section_le_monde = SoupStrainer(class_="teaser teaser--inline-picture  ")

    mentions = []
    years = []

    options = webdriver.ChromeOptions()
    options.add_argument("--headless=new") # for Chrome >= 109

    driver = webdriver.Chrome(options=options)

    # Provoquer demande d'acceptation de cookies
    test_url = f"https://lemonde.fr/recherche/?search_keywords={'test'}&start_at=1%2F1%2F{2000}&end_at=1%2F1%2F{2001}&search_sort=relevance_desc"
    driver.get(test_url)
    driver.add_cookie({"name": "euconsent-v2", "value": "CQMgjEAQMgjEAFzABBFRBbFsAP_gAAAAAAqII7MB7C7OQSFicX51AOsESY1ewkBSoEAgBAABgQgBABgQsIwRkGAKIADAAqAKAAIAImRBIQAtGAjABAAAQIAAASCEAECAABAAJKAAAEAQAAEACAgACQEAAAAAgEAAAAQAgAIAAEooQERAAgAgLAAAIAABAIAEAgAIAAAAAAAAAAAAAGAAAAAAAAAAAAAAAAAAEEBAARBQiIACwICQggDCCBACoIQAIAAAQAAAAwAAAAAwIUAYACDABAAAAAAAEAAAAARAAgAAAgAQgAAAAIEAAAAAAAAAAAAgAABAAAAAAAAAAAAQAAIAAAAAAAAAAAAIAQAAAAAAAICAAAAAQFgAAAAAAgEAAAAAAAAAAAAAAAAAAACAMEgAwABBAQdABgACCAhCADAAEEBCUAGAAIICFIAMAAQQELQAYAAggIA", "sameSite": "Lax"})
    driver.add_cookie({"name": "lmd_consent", "value": "%7B%22userId%22%3A%22e5912aad-e232-4533-9da0-cdd43a427087%22%2C%22timestamp%22%3A%221739010630.749987654%22%2C%22version%22%3A1%2C%22cmpId%22%3A371%2C%22displayMode%22%3A%22cookiewall%22%2C%22purposes%22%3A%7B%22analytics%22%3Atrue%2C%22ads%22%3Atrue%2C%22personalization%22%3Atrue%2C%22mediaPlatforms%22%3Atrue%2C%22social%22%3Atrue%7D%7D", "sameSite": "Lax"})
    time.sleep(3)

    # commencer scrapping
    for year in range(start_year,end_year+1):
        url = f"https://lemonde.fr/recherche/?search_keywords={subject}&start_at=1%2F1%2F{year}&end_at=1%2F1%2F{year+1}&search_sort=relevance_desc"

        try:
            driver.get(url)
        except:
            print('timeout')
            driver.execute_script("window.stop()")        
        
        time.sleep(sleep)
        
        # Get the page source after JavaScript has rendered the content
        page_source = driver.page_source
        site = bs4.BeautifulSoup(page_source, 'html.parser',parse_only=only_page_count_le_monde)
        site = str(site)
        filter_results = re.findall(pattern=r'(?<=page\=)\d+(?=\")',string=str(site))
        if len(filter_results) > 0:
            number_of_pages = max([int(x) for x in filter_results])
        else:
            number_of_pages = 1


        # number of sections
        article_sections = bs4.BeautifulSoup(page_source, 'html.parser',parse_only=only_article_section_le_monde)
        number_of_articles = len(re.findall(r'\<section class\=',str(article_sections)))
        mentions.append(number_of_pages * number_of_articles)
        years.append(year)
    driver.close()
    return pd.DataFrame({'Year':years, 'Mentions': mentions}).set_index('Year')



### Parcourir Libération.fr

In [5]:
def get_liberation_mentions(subject, start_year=2010,end_year = 2024,sleep=3):
    subject = subject.replace(" ","+")

    url = f'https://www.liberation.fr/recherche/?query={subject}'

    mentions = []
    years = []

    # Libération
    options = webdriver.ChromeOptions()
    #options.add_argument("--headless=new") # for Chrome >= 109
    driver = webdriver.Chrome(options=options)
    driver.get(url)
    time.sleep(3)
    driver.switch_to.frame(driver.find_element(By.ID,"sp_message_iframe_1223720"))
    cookie_notice = driver.find_element(By.ID, 'notice')
    cookie_notice.find_element(By.XPATH,'//*[@title="Accepter et continuer"]').click()

    #print(driver.find_element(By.CLASS_NAME, 'message-component message-button no-children focusable start-focus sp_choice_type_11'))
    driver.switch_to.default_content()
    tries = 0
    time.sleep(sleep)
    for year in range(start_year,end_year+1):
        #print(driver.find_elements(By.ID, "datepicker_from"))
        while (driver.find_elements(By.ID, "datepicker_from") == []):
            if tries >= 3:  
                raise Exception(ConnectionError)
            driver.refresh()
            time.sleep(sleep)
            tries += 1
        datepicker_from = driver.find_element(By.ID,"datepicker_from")
        datepicker_from.clear()

        datepicker_from.send_keys(f"01/01/{year}")
        datepicker_from.send_keys(Keys.TAB)
        driver.find_element(By.ID,"datepicker_to").send_keys(f"01/01/{year+1}")
        driver.execute_script('searchPage.applyDateRange();')
        time.sleep(1)

        only_filter_liberation = SoupStrainer(id="resultdata")
        page_source = driver.page_source
        site = bs4.BeautifulSoup(page_source, 'html.parser',parse_only=only_filter_liberation)
        pattern = r'(?<=Affichage de )\d+(?= résultats)'

        filter_results = re.findall(pattern,str(site))
        if len(filter_results) > 0:
            number_of_mentions = filter_results[0]
        else:
            number_of_mentions = 0
        years.append(year)
        mentions.append(int(number_of_mentions))
    driver.close()
    df = pd.DataFrame({'Year':years, 'Mentions': mentions}).set_index('Year')
    return df


### Parcourir TAZ

In [6]:
only_search_result_taz = SoupStrainer(class_='typo-teaser-text-bold  mgb-medium')

def do_taz_search(subject, year_from, month_from, month_to):
    if month_from == month_to:
        year_to = year_from + 1
    else:
        year_to = year_from
    url = f'https://taz.de/!s={subject}&eTagAb={year_from}-{month_from}-01&eTagBis={year_to}-{month_to}-01/'
    table_request_text = request.urlopen(url).read()
    site = bs4.BeautifulSoup(table_request_text, "html.parser",parse_only=only_search_result_taz)
    site = str(site)

    pattern = r'(?<=von )\d+(?= \<\/p\>)'
    filter_results = re.findall(pattern, site)
    if len(filter_results) > 0:
        number_of_mentions = filter_results[0]
    else:
        number_of_mentions = 0
    
    return int(number_of_mentions)

def get_taz_mentions(subject, start_year=2010,end_year = 2024,sleep=3):
    subject = subject.replace(" ","+")

    mentions = []
    years = []

   
    for year in range(start_year,end_year+1):
        month = 1
        following_month = month # only relevant for monthly search
        number_of_mentions = do_taz_search(subject, year, month, following_month)
        if(number_of_mentions >= 1000):
            # max limit reached, do monthly search
            print(f'Starting {year} monthly serach')
            number_of_mentions = 0
            for month in range(1,12):
                number_of_mentions += do_taz_search(subject, year, month, month +1)


        years.append(year)
        mentions.append(int(number_of_mentions))
        time.sleep(sleep)

    return pd.DataFrame({'Year':years, 'Mentions': mentions}).set_index('Year')

In [ ]:
# fonction qui permet la recherche des quatre journaux avec les même paramètres
def get_mentions(subjects, start_year=2010,end_year = 2024,sleep=3):
    subjects_original = subjects.copy()
    subjects = [urllib.parse.quote_plus(subject) for subject in subjects]
    print (subjects)
    print('Getting FAZ')
    try:
        mentions_faz_df = get_faz_mentions(subjects[0], start_year, end_year, sleep)
    except:
        mentions_faz_df = get_faz_mentions(subjects[0], start_year, end_year, sleep)

    print('Getting TAZ')
    try:
        mentions_taz_df = get_taz_mentions(subjects[0], start_year, end_year, sleep)
    except:
        mentions_taz_df = get_taz_mentions(subjects[0], start_year, end_year, sleep)

    print('Getting LeMonde')
    try:
        mentions_lemonde_df = get_lemonde_mentions(subjects[1],start_year, end_year,sleep)
    except:
        mentions_lemonde_df = get_lemonde_mentions(subjects[1],start_year, end_year,sleep)
        
    print('Getting Liberation')
    try:
        mentions_liberation_df = get_liberation_mentions(subjects[1],start_year, end_year,sleep)
    except:
        mentions_liberation_df = get_liberation_mentions(subjects[1],start_year, end_year,sleep)
        

    print('Done')

    mentions = mentions_faz_df.copy()
    mentions['taz'] = mentions_taz_df['Mentions']

    mentions['liberation'] = mentions_liberation_df['Mentions']
    mentions['lemonde'] = mentions_lemonde_df['Mentions']
    mentions = mentions.rename(columns={'Mentions':'faz'})
    with open(f'../data/ScrappingSujets/{subjects_original[0]}_{subjects_original[1]}_{start_year}_{end_year}.pckl', 'wb') as f:
        mentions.to_pickle(f)
    
    print(mentions)

    return mentions


In [ ]:
# fonction adaptée pour les journaux allemands
def get_mentions_only_de_array(subjects_dict,id, start_year=2010,end_year = 2024,sleep=3):
    first = True
    years = sorted(subjects_dict)
    # find the first term
    if start_year in years:
        first_term = subjects_dict[start_year]
    elif 0 in years:
        first_term = subjects_dict[0]
    else: 
        for year in range(1900,start_year):
            if year in years:
                first_term = subjects_dict[year]

    years = [year for year in years if year != 0 and year >= start_year and year <= end_year ]
    if len(years) == 0:
        intervals = [(start_year,end_year)]
    else:
        intervals = [(start_year,years[0]-1)]+[(year, years[i+1]-1) for i,year in enumerate(years[:-1])]+[(years[-1],end_year)]
    intervals = [interval for interval in intervals if interval[0] <= interval[1]]
    dfs = []
    print(intervals)
    for interval in intervals:
        if first:
            subjects = first_term
        else:
            subjects = subjects_dict[interval[0]]
        
        first = True
        for subject in subjects:
            subject = urllib.parse.quote_plus(subject) 
            print(subject)
            print(interval)
            print('Getting FAZ')

            if  first: 
                try:
                    mentions_faz_df = get_faz_mentions(subject, interval[0], interval[1], sleep)
                except:
                    mentions_faz_df = get_faz_mentions(subject, interval[0], interval[1], sleep)

                print('Getting TAZ')
                try:
                    mentions_taz_df = get_taz_mentions(subject, interval[0], interval[1], sleep)
                except:
                    mentions_taz_df = get_taz_mentions(subject, interval[0], interval[1], sleep)
                first = False
            else:
                try:
                    mentions_faz_df += get_faz_mentions(subject, interval[0], interval[1], sleep)
                except:
                    mentions_faz_df += get_faz_mentions(subject, interval[0], interval[1], sleep)

                print('Getting TAZ')
                try:
                    mentions_taz_df += get_taz_mentions(subject, interval[0], interval[1], sleep)
                except:
                    mentions_taz_df += get_taz_mentions(subject, interval[0], interval[1], sleep)

        print("Mentions FAZ", mentions_faz_df)
        print("Mentions TAZ", mentions_taz_df)
        mentions = mentions_faz_df.copy()
        mentions['taz'] = mentions_taz_df['Mentions']
        mentions = mentions.rename(columns={'Mentions':'faz'})
        dfs.append(mentions)
            
    print('Done')

    final_df = pd.concat(dfs)
    with open(f'../../data/ScrappingJournaux/de/{id}_{start_year}_{end_year}.pckl', 'wb') as f:
         final_df.to_pickle(f)

    

    return final_df


## Mots tirés du sondage, question : "Quel est le plus grand problème en Allemagne actuellement ?"

In [12]:
data = {
    1: {
        0: ["Renten", "Alterssicherung"],
    },
    5: {
        1986: ["Inflation"],
        1989: ["Wirtschaft", "Steuern", "Inflation"],
        1990: ["Steuern", "Steuererhöhung", "Gebühren", "Abgaben"],
        1993: ["Steuern", "Steuererhöhung", "Autobahngebühr"],
        1994: ["Steuern", "Steuererhöhung"],
        2006: ["Lebenshaltungskosten",  "Benzinpreise", "Teuro", "Inflation"],
        2013: ["Lebenshaltungskosten",  "Benzinpreise", "Teuro", "Inflation"],
        2015: ["Lebenshaltungskosten",  "Benzinpreise", "Inflation"],
    },
    10: {
        1986: ["Steuern", "Steuerreform"],
        1989: ["Wirtschaft", "Steuern", "Inflation"],
        1990: ["Steuern", "Steuererhöhung", "Gebühren", "Abgaben"],
        1993: ["Steuern", "Steuererhöhung", "Autobahngebühren"],
        1994: ["Steuern", "Steuererhöhungen"],
        2006: ["Steuern", "Steuererhöhungen"],
        2014: ["Steuern", "Steuererhöhungen", "Steuerhinterziehung"],
        2015: ["Steuern", "Steuererhöhungen", "Steuerhinterziehung"],
    },
    13: {
        1990: ["Wirtschaftspolitik"],
        1993: ["Wirtschaftspolitik", "Wirtschaftslage"],
        1994: ["Wirtschaftswachstum", "Wirtschaftslage"],
        2008: ["Rezession"],
        2011: ["Wirtschaftslage", "Rezession"],
        2012: ["Wirtschaftslage"],
        2020: ["Wirtschaftslage", "Konjunkturpaket", "Corona-Hilfen"],
    },
    14: {
        0: ["Arbeitslosigkeit", "Arbeitsplätze", "Ausbildungsplätze"],  # Entrée uniquement pour l'année de base
    },
    15: {
        1989: ["Umweltschutz"],
        1994: ["Umweltschutz", "Ozon"],
        1995: ["Umweltschutz", "Ölplattform Brent Spar"],
        2005: ["Umweltschutz", "Atomtransport"],
        2006: ["Umweltschutz", "Hochwasser", "Flut"],
        2007: ["Umweltschutz", "Klima", "Klimawandel"],
        2008: ["Erneuerbare Energien"],
        2012: ["Umweltschutz", "Klima", "Klimawandel", "Erneuerbare Energien", "Energiewende"],
        2013: ["Umweltschutz", "Klimawandel"],
        2016: ["Umweltschutz", "Klimawandel", "Klimagipfel"],
        2019: ["Umweltschutz", "Klimawandel", "Schutz von Insekten"],
        2020: ["Umweltschutz", "Klimawandel", "Artenschutz"],
    },
    19: {
        0: ["Kindergartenplätze"],  # Entrée uniquement pour l'année de base
        1996: ["Familie", "Kinder", "Jugend"],
    },
    20: {
        1989: ["Gesundheitsreform"],
        1990: ["Gesundheitswesen", "Pflegenotstand"],
        1994: ["Gesundheitswesen"],
        2006: ["Gesundheitswesen", "Pflegeversicherung"],
        2011: ["Gesundheitswesen", "Gesundheitsreform", "Pflegeversicherung"],
        2012: ["Gesundheitswesen", "Gesundheitspolitik", "Pflegeversicherung"],
        2014: ["Gesundheitswesen", "Gesundheitspolitik", "Pflegeversicherung"],
        2022: ["Gesundheitswesen", "Gesundheitspolitik", "Pflegeversicherung"],
    },
    23: {
        1986: ["Innere Sicherheit", "Kriminalität", "Terrorismus", "härtere Strafen", "Kronzeugenregelung"],
        1989: ["Innere Sicherheit", "Ruhe und Ordnung"],
        1990: ["Ruhe und Ordnung", "Kriminalität"],
        2008: ["Jugendkriminalität"],
        2011: ["Kriminalität", "Jugendkriminalität", "Ruhe und Ordnung"],
        2016: ["Kriminalität", "Ruhe und Ordnung", "mehr Polizei"],
        2019: ["Kriminalität", "Ruhe und Ordnung", "mehr Polizei", "Innere Sicherheit"],
        2020: ["Kriminalität", "Ruhe und Ordnung", "mehr Polizei", "Innere Sicherheit"],
        2021: ["Kriminalität", "Ruhe und Ordnung", "mehr Polizei", "Innere Sicherheit"],
    },
    27: {
        1986: ["Politische Moral", "Spendenaffären", "Flick"],
        1993: ["Partei Affäre", "Politikverdruß"],
        2001: ["Partei Affäre", "Politikverdruß", "Politik allgemein"],
        2002: ["Partei Affäre", "Politikverdruß"],
        2005: ["Partei Affäre", "Politikverdruß","Spenden affäre"],
        2006: ["Partei Affäre", "Politikverdruß"],
    },
    30: {
        1989: ["Bildungspolitik", "Schulpolitik"],
        1997: ["Studentenproteste", "Bildung"],
        1999: ["Schule", "Bildung"],
        2002: ["Schule", "Bildung", "Pisa"],
    },
    53: {
        1986: ["Ausländer", "Asylanten", "Aussiedler"],
        1989: ["Asylanten", "Asyl"],
        2005: ["Asylanten", "Asyl"],
        2018: ["Ausländerfeindlichkeit"],
    },
    54: {
        1989: ["Ausländer"],
        2002: ["Ausländer", "doppelte Staatsbürgerschaft"],
        2005: ["Ausländer", "Zuwanderung", "Asylanten", "Asyl"],
        2006: ["Ausländer", "Zuwanderung", "Integration", "Asyl"],
        2016: ["Ausländer", "Zuwanderung", "Integration", "Asyl", "Flüchtlinge"],
        2018: ["Ausländer", "Zuwanderung", "Integration", "Asyl", "Flüchtlinge", "BAMF"],
        2019: ["Ausländer", "Zuwanderung", "Integration", "Asyl", "Flüchtlinge"],
    },
    55: {
        1990: ["EU Krise", "Binnenmarkt"],
        1998: [ "EU Krise", "Binnenmarkt"],
        1999: [ "EU Krise", "Euro Krise"],
        2002: [ "EU Krise", "Euro Krise"],
        2005: [ "EU Krise", "Türkei-Beitritt"],
        2012: ["EU Krise",  "Euro Krise" ],
        2013: ["EU Krise",  "Euro Krise"  ],
        2014: ["EU Krise",  "Euro Krise"   ],
        2015: ["EU Krise",  "Euro Krise"   ],
        2017: ["EU Krise",  "Euro Krise", "Brexit"  ],
        2021: ["EU Krise",  "Euro Krise", "Brexit"   ],
    },
    56: {
        1992: ["Staatsverschuldung"],
        2012: ["Staatsverschuldung", "Verschuldung der Bundesländer"],
        2022: ["Staatsverschuldung", "Verschuldung der Bundesländer"],
    },
    58: {
        1990: ["Rechtsradikale", "Rechtsextreme"],
        1991: ["NPD"],
        1989: ["Rechtsextremismus", "Republikaner", "DVU"],
        2002: ["Rechtsradikale", "Rechtsextreme", "NPD"],
        2004: ["Rechtsradikale", "Rechtsextreme", "NPD", "Antisemitismus"],
        2012: ["Rechtsextreme", "NPD", "Antisemitismus", "Verfassungsschutz"],
        2016: ["Rechtsextreme", "NPD", "Antisemitismus"],
        2017: ["Rechtsradikale", "NPD"],
        2018: ["Rechtsradikale", "Rechtsextremismus", "Rechtspopulismus"],
        2019: ["Rechtsradikale", "Rechtsextremismus", "Rechtspopulismus"],
        2020: ["Rechtsradikale", "Rechtsextremismus", "Rechtspopulismus"],
        2021: ["Rechtsradikale", "Rechtsextremismus", "Rechtspopulismus"],
    },
    77: {
        1998: ["soziales Gefälle", "arm-reich"],
        2006: ["soziales Gefälle", "arm-reich", "Unterschicht"],
        2014: ["soziales Gefälle", "Gerechtigkeit", "arm-reich", "Unterschicht"],
    },
}
data =  {
    17: {
        0: ["Finanzkrise","Bankenkrise"]
    },
    117: {
        0:["Coronavirus"],
        2020: ["Coronavirus", "neues Virus aus China", "Ausbreitung Virus"],
        2020: ["Coronavirus", "Ausbreitung Virus"],
        2021: ["Coronavirus", "Impfen", "Folgen für verschiedene Bereiche"],
        2022: ["Coronavirus", "Impfen", "Folgen pour verschiedene Bereiche"],
    }
}


In [ ]:
# Start Scrapping
for key, subjects_dict in data.items(): 
    get_mentions_only_de_array(subjects_dict,key,start_year=2000, end_year=2024) 